# Training-Only Model Experiments

## 1. Introduction

This notebook compares two small, explainable logistic-regression candidates using **only the 56,155 real training respondents**. It does not open or read the held-out test CSV.

**Success rule:** set each fold's decision threshold using inner out-of-fold training predictions with a recall target of at least 85%. The outer folds estimate how well that target carries to new respondents. Rank candidates by mean outer-fold specificity (fewer false alarms), and report balanced-sample precision and precision adjusted to the approximate 14% prevalence. The final held-out test is reserved for a single comparison after the CV finalist and threshold are fixed.

We compare: (1) the current M1 model and (2) the current model with Education and Physical Activity added. The extra features appeared in an earlier project model and are a small, interpretable feature-set check. Synthetic rows are excluded because the previous shared-test comparison found no meaningful gain.

Thresholds are set separately inside each outer fold from inner out-of-fold predictions. Outer validation respondents are used only to estimate performance. The 85% recall threshold is therefore a training-time target, not a guarantee that every outer fold or future population will reach exactly 85%.

## 2. Load the training data and the current M1 feature list

We use the saved 70k M1 bundle only to obtain its current feature list. The model itself is refit in every fold. The test file is not referenced in this notebook.

In [1]:
# Import the tools for data handling, model pipelines and cross-validation.
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Resolve repository paths from this notebook's folder.
ROOT = Path("../..").resolve()
TRAIN_CSV = ROOT / "data/05_cmb_train_test_for_models/diabetes_model_train.csv"
M1_PATH = ROOT / "application/models/logreg_70k.joblib"
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Freeze the study settings before comparing candidates.
TARGET = "Diabetes_binary"
RANDOM_STATE = 42
TARGET_RECALL = 0.85
REAL_PREVALENCE = 0.14
OUTER_FOLDS = 5
INNER_FOLDS = 5
NON_BINARY = {"BMI", "GenHlth", "MentHlth", "PhysHlth", "Age", "Education", "Income"}

print("Settings ready; only the training CSV and M1 bundle will be loaded.")

Settings ready; only the training CSV and M1 bundle will be loaded.


## 3. Confirm data and define the candidate models

All candidates use the same real respondents and seeded folds. Scaling is fitted inside each training fold. The expanded-feature candidate adds only Education and Physical Activity; its test must show a cross-validated gain before it could replace M1.

In [2]:
# Load the real balanced training rows and the current M1 bundle.
train_df = pd.read_csv(TRAIN_CSV)
m1_bundle = joblib.load(M1_PATH)
M1_FEATURES = list(m1_bundle["features"])

# Check the expected training partition and ensure synthetic markers are absent.
if len(train_df) != 56_155:
    raise ValueError(f"Expected 56,155 training rows, found {len(train_df):,}.")
if "is_synthetic" in train_df.columns:
    raise ValueError("This experiment must use real 70k training rows only.")
if TARGET not in train_df.columns:
    raise ValueError(f"Training CSV is missing {TARGET!r}.")
if train_df[TARGET].value_counts().to_dict() != {0: 28_077, 1: 28_078}:
    raise ValueError("Training class counts differ from the expected real training split.")
if any(feature not in train_df.columns for feature in M1_FEATURES):
    raise ValueError("The saved M1 feature list is not present in the training CSV.")

# Add only previously used, survey-supported features for the feature experiment.
ADDED_FEATURES = ["Education", "PhysActivity"]
if any(feature not in train_df.columns for feature in ADDED_FEATURES):
    raise ValueError("An added candidate feature is missing from the training CSV.")

# Each candidate varies one modeling choice while keeping the data and target fixed.
CANDIDATES = {
    "M1_current": {"label": "Current M1: unpenalized logistic regression",
                   "features": M1_FEATURES, "C": np.inf},
    "Add_Education_PhysActivity": {"label": "M1 features + Education + Physical Activity",
                                  "features": M1_FEATURES + ADDED_FEATURES, "C": np.inf},
}

class_summary = pd.DataFrame({
    "Set": ["Real training rows"],
    "Rows": [len(train_df)],
    "No diagnosed diabetes": [(train_df[TARGET] == 0).sum()],
    "Reported diagnosed diabetes": [(train_df[TARGET] == 1).sum()],
    "M1 features": [len(M1_FEATURES)],
    "Experiment features": [len(ADDED_FEATURES)],
})
display(class_summary.style.hide(axis="index"))
print("Training data and candidate definitions are ready.")

Set,Rows,No diagnosed diabetes,Reported diagnosed diabetes,M1 features,Experiment features
Real training rows,56155,28077,28078,14,2


Training data and candidate definitions are ready.


## 4. Define the pipeline, training-only threshold, and metrics

Recall is the share of positive cases found; specificity is the share of negative cases correctly left unflagged. Precision is shown on the balanced sample and recalculated at the assumed 14% prevalence. That prevalence adjustment is approximate and is not a personalized probability.

In [3]:
# Fit standardization only on the rows passed to each pipeline fit.
def make_model(features, C):
    scaled = [name for name in features if name in NON_BINARY]
    prep = ColumnTransformer(
        [("scale", StandardScaler(), scaled)],
        remainder="passthrough",
        verbose_feature_names_out=False,
    )
    return Pipeline([
        ("prep", prep),
        ("model", LogisticRegression(C=C, max_iter=2000, random_state=RANDOM_STATE)),
    ])

# Select the highest threshold whose inner out-of-fold training recall meets the target.
def threshold_for_recall(y_true, probabilities):
    _, recall, thresholds = precision_recall_curve(y_true, probabilities)
    eligible = thresholds[recall[:-1] >= TARGET_RECALL]
    if len(eligible) == 0:
        raise RuntimeError("No inner out-of-fold threshold reaches the 85% recall target.")
    return float(eligible.max())

# Summarize performance at one already-fixed threshold.
def score_at_threshold(y_true, probabilities, threshold):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    recall = tp / (tp + fn)
    specificity = tn / (tn + fp)
    flagged_share = recall * REAL_PREVALENCE + (1 - specificity) * (1 - REAL_PREVALENCE)
    return {
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, predictions),
        "Recall": recall,
        "Specificity": specificity,
        "Precision (balanced)": precision_score(y_true, predictions, zero_division=0),
        "Precision @ 14% prevalence": recall * REAL_PREVALENCE / flagged_share if flagged_share else 0.0,
        "Share flagged @ 14% prevalence": flagged_share,
        "F1": f1_score(y_true, predictions, zero_division=0),
        "ROC AUC": roc_auc_score(y_true, probabilities),
        "Brier score": brier_score_loss(y_true, probabilities),
        "False negatives": int(fn),
        "False positives": int(fp),
    }

print("Pipeline and scoring rules are ready; the held-out test set is still unopened.")

Pipeline and scoring rules are ready; the held-out test set is still unopened.


## 5. Run nested, training-only cross-validation

The outer folds estimate performance. For each outer fold, an inner cross-validation produces out-of-fold probabilities on only the outer-training respondents and sets that fold’s threshold. The outer-validation fold is then scored once. This gives each candidate a fair comparison on the same held-out training folds without consulting the final test set.

In [4]:
# Prepare features and labels from the real training partition only.
X_all = train_df.drop(columns=TARGET)
y_all = train_df[TARGET].astype(int)
outer_cv = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
fold_rows = []

# Reuse the same outer respondent folds for all candidates.
for outer_fold, (outer_train_idx, outer_valid_idx) in enumerate(outer_cv.split(X_all[M1_FEATURES], y_all), start=1):
    X_outer_train = X_all.iloc[outer_train_idx]
    y_outer_train = y_all.iloc[outer_train_idx]
    X_outer_valid = X_all.iloc[outer_valid_idx]
    y_outer_valid = y_all.iloc[outer_valid_idx]

    # Inner folds are drawn only from this outer training portion.
    inner_cv = StratifiedKFold(
        n_splits=INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE + outer_fold
    )

    for candidate_id, spec in CANDIDATES.items():
        features = spec["features"]

        # Estimate a threshold from inner out-of-fold predictions only.
        inner_oof = cross_val_predict(
            make_model(features, spec["C"]),
            X_outer_train[features],
            y_outer_train,
            cv=inner_cv,
            method="predict_proba",
            n_jobs=1,
        )[:, 1]
        fold_threshold = threshold_for_recall(y_outer_train, inner_oof)

        # Refit on the outer training rows, then score its untouched validation fold.
        outer_model = make_model(features, spec["C"]).fit(
            X_outer_train[features], y_outer_train
        )
        outer_probabilities = outer_model.predict_proba(X_outer_valid[features])[:, 1]
        inner_recall_at_threshold = (
            ((inner_oof >= fold_threshold) & (y_outer_train.to_numpy() == 1)).sum()
            / (y_outer_train.to_numpy() == 1).sum()
        )
        fold_rows.append({
            "Candidate": candidate_id,
            "Description": spec["label"],
            "Outer fold": outer_fold,
            "Validation rows": len(outer_valid_idx),
            "Inner OOF recall at threshold": inner_recall_at_threshold,
            **score_at_threshold(y_outer_valid, outer_probabilities, fold_threshold),
        })

    print(f"Completed outer fold {outer_fold}/{OUTER_FOLDS}; thresholds came from inner training folds.")

fold_results = pd.DataFrame(fold_rows)
print("Nested cross-validation completed using training respondents only.")

Completed outer fold 1/5; thresholds came from inner training folds.


Completed outer fold 2/5; thresholds came from inner training folds.


Completed outer fold 3/5; thresholds came from inner training folds.


Completed outer fold 4/5; thresholds came from inner training folds.


Completed outer fold 5/5; thresholds came from inner training folds.
Nested cross-validation completed using training respondents only.


## 6. Apply the CV ranking rule

A candidate is eligible when inner training OOF predictions in all five outer folds support a threshold meeting the 85% recall target. Mean outer-fold recall estimates generalization and may fall slightly below 85%. Rank by mean outer-fold specificity, then report precision and fold variation. If the feature candidate does not improve this training-only comparison, keep M1 and leave the held-out test for a future, justified candidate.

In [5]:
# Summarize each candidate across the same five outer validation folds.
summary_rows = []
for candidate_id, rows in fold_results.groupby("Candidate", sort=False):
    spec = CANDIDATES[candidate_id]
    summary_rows.append({
        "Candidate": candidate_id,
        "Description": spec["label"],
        "Features": len(spec["features"]),
        "Mean threshold": rows["Threshold"].mean(),
        "Mean inner OOF recall": rows["Inner OOF recall at threshold"].mean(),
        "Outer folds with inner OOF recall >= 85%": int((rows["Inner OOF recall at threshold"] >= TARGET_RECALL).sum()),
        "Mean outer CV recall": rows["Recall"].mean(),
        "SD outer CV recall": rows["Recall"].std(ddof=1),
        "Outer folds with validation recall >= 85%": int((rows["Recall"] >= TARGET_RECALL).sum()),
        "Mean specificity": rows["Specificity"].mean(),
        "SD specificity": rows["Specificity"].std(ddof=1),
        "Mean precision (balanced)": rows["Precision (balanced)"].mean(),
        "Mean precision @ 14% prevalence": rows["Precision @ 14% prevalence"].mean(),
        "Mean share flagged @ 14% prevalence": rows["Share flagged @ 14% prevalence"].mean(),
        "Mean ROC AUC": rows["ROC AUC"].mean(),
        "Mean Brier score": rows["Brier score"].mean(),
    })
cv_summary = pd.DataFrame(summary_rows)
cv_summary["Inner OOF target achieved"] = cv_summary["Outer folds with inner OOF recall >= 85%"] == OUTER_FOLDS
eligible = cv_summary.loc[cv_summary["Inner OOF target achieved"]]

# Rank eligible candidates by outer-fold specificity; use 14%-prevalence precision only as a secondary tie-break.
if not eligible.empty:
    ordered = eligible.sort_values(
        ["Mean specificity", "Mean precision @ 14% prevalence"], ascending=False
    )
    finalist_id = ordered.iloc[0]["Candidate"]
    selection_status = ("The finalist had a training-derived threshold reaching the 85% inner OOF recall target "
                        "in every outer fold; ranking uses outer-fold specificity.")
else:
    ordered = cv_summary.sort_values(["Mean inner OOF recall", "Mean specificity"], ascending=False)
    finalist_id = ordered.iloc[0]["Candidate"]
    selection_status = ("No candidate reached the 85% recall target on inner OOF predictions in every fold; "
                        "the top candidate is carried forward for diagnosis only.")

cv_summary["Rank"] = np.nan
for rank, candidate_id in enumerate(ordered["Candidate"], start=1):
    cv_summary.loc[cv_summary["Candidate"] == candidate_id, "Rank"] = rank
cv_summary = cv_summary.sort_values("Rank")
display(cv_summary.round(4).style.hide(axis="index"))
print(selection_status)
print(f"Training-only CV finalist: {finalist_id} — {CANDIDATES[finalist_id]['label']}.")

# Save the training-only evidence for the final-evaluation notebook.
cv_summary.to_csv(RESULTS_DIR / "training_cv_candidate_summary.csv", index=False)
pd.DataFrame(fold_rows).to_csv(RESULTS_DIR / "training_cv_fold_results.csv", index=False)
pd.DataFrame([{
    "finalist_id": finalist_id,
    "inner_oof_target_reached": bool(cv_summary.loc[cv_summary["Candidate"] == finalist_id, "Inner OOF target achieved"].iloc[0]),
    "selection_status": selection_status,
}]).to_csv(RESULTS_DIR / "cv_finalist.csv", index=False)
print("Saved training-only CV summary, fold metrics and finalist selection under results/.")

Candidate,Description,Features,Mean threshold,Mean inner OOF recall,Outer folds with inner OOF recall >= 85%,Mean outer CV recall,SD outer CV recall,Outer folds with validation recall >= 85%,Mean specificity,SD specificity,Mean precision (balanced),Mean precision @ 14% prevalence,Mean share flagged @ 14% prevalence,Mean ROC AUC,Mean Brier score,Inner OOF target achieved,Rank
M1_current,Current M1: unpenalized logistic regression,14,0.396500,0.850000,5,0.849800,0.003800,3,0.607200,0.005300,0.683900,0.260500,0.456800,0.807300,0.178500,True,1.000000
Add_Education_PhysActivity,M1 features + Education + Physical Activity,16,0.396600,0.850000,5,0.849300,0.003100,2,0.607000,0.005200,0.683700,0.260300,0.456900,0.807300,0.178500,True,2.000000


The finalist had a training-derived threshold reaching the 85% inner OOF recall target in every outer fold; ranking uses outer-fold specificity.
Training-only CV finalist: M1_current — Current M1: unpenalized logistic regression.
Saved training-only CV summary, fold metrics and finalist selection under results/.


## 7. Training-only conclusion

This run compares current M1 with the limited feature addition. Use the outer-fold recall and specificity in the table to decide whether the feature change merits further work. If it does not improve the CV comparison, keep M1 as the app model and do not open the held-out test for this candidate. Do not tune thresholds or features using the test set.